<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

In [1]:
# ==============================================================================
# Part 1: Environment Setup and Dataset Loading
# ==============================================================================

# 1.1: Install necessary libraries
# - peft: Parameter-Efficient Fine-Tuning library for methods like LoRA.
# - transformers: Hugging Face's library for state-of-the-art models (like PaliGemma).
# - datasets: For loading and processing datasets from the Hugging Face Hub.
# - evaluate & rouge_score: For calculating evaluation metrics.
# - bitsandbytes: For model quantization (like 8-bit loading) to save memory.
# - huggingface_hub: For interacting with the Hugging Face Hub (e.g., logging in and uploading).
!pip install -q peft transformers datasets evaluate bitsandbytes rouge_score huggingface_hub

# 1.2: Import required modules
import os                     # For interacting with the operating system (e.g., creating directories).
import numpy as np            # For numerical operations (though less used here, it's good practice).
import logging                # To display informative messages during the script's execution.
from PIL import Image         # For handling and processing image files.
import torch                  # The core deep learning framework.
import random                 # For selecting random examples for qualitative evaluation.
import json                   # To save the detailed evaluation results to a file.

from datasets import load_dataset # Function to load datasets from the Hugging Face Hub.
from peft import LoraConfig, get_peft_model # For setting up and applying LoRA to the model.
from transformers import (
    PaliGemmaProcessor,             # The processor for PaliGemma, handles both image and text preprocessing.
    PaliGemmaForConditionalGeneration, # The PaliGemma model architecture.
    Trainer,                        # A class that simplifies the model training and evaluation loop.
    TrainingArguments,              # A class to configure all the hyperparameters for the Trainer.
    BitsAndBytesConfig,             # A class to configure model quantization.
)
import evaluate               # The main library for loading evaluation metrics.
from huggingface_hub import notebook_login # A function to log into the Hugging Face Hub from a notebook.

# 1.3: Setup Logging and Device Configuration
# This configures logging to show INFO level messages, making the process more transparent.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check for a CUDA-enabled GPU and set the device accordingly.
# Training is significantly faster on a GPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
    logger.info(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    logger.info("GPU not available, using CPU instead.")

# 1.4: Load and Split the Dataset
logger.info("Loading the clevr_cogen_a_train dataset...")
# We load the first 50% of the 'train' split from the specified dataset on the Hugging Face Hub.
full_subset = load_dataset("leonardPKU/clevr_cogen_a_train", split="train[:20%]")

# We split the loaded data into a training set (90%) and a testing set (10%).
# The `seed` ensures this split is reproducible.
split_datasets = full_subset.train_test_split(test_size=0.1, seed=42)

# Assign the final datasets for training and testing.
train_dataset = split_datasets["train"]
test_dataset = split_datasets["test"]

logger.info(f"Training dataset size: {len(train_dataset)}")
logger.info(f"Testing dataset size: {len(test_dataset)}")

/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:__main__:Using device: NVIDIA GeForce RTX 3090


INFO:__main__:Loading the clevr_cogen_a_train dataset...


INFO:__main__:Training dataset size: 12600


INFO:__main__:Testing dataset size: 1400


In [2]:
# ==============================================================================
# Part 2: Model and LoRA Configuration
# ==============================================================================

# 2.1: Load the Processor
model_id = "./paligemma-3b-mix-224"
logger.info(f"Loading processor from {model_id}...")
# The processor is a crucial component that handles both text tokenization and image preprocessing
# in a way that is compatible with the PaliGemma model.
processor = PaliGemmaProcessor.from_pretrained(model_id)

# 2.2: Configure BitsAndBytes for 8-bit Quantization
# Quantization reduces the model's memory footprint by using lower precision numbers (8-bit integers).
# This allows us to fine-tune a large model on a single consumer GPU.
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,      # Enable 8-bit loading.
    llm_int8_threshold=6.0, # A parameter for the quantization algorithm.
)

# 2.3: Load the PaliGemma Model with Quantization
logger.info("Loading PaliGemma model in 8-bit precision...")
# We load the pre-trained PaliGemma model.
# - `device_map="auto"`: Automatically distributes the model layers across available hardware (GPU/CPU).
# - `quantization_config=bnb_config`: Applies the 8-bit quantization defined above.
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

# 2.4: Configure LoRA (Low-Rank Adaptation)
logger.info("Configuring LoRA for efficient fine-tuning...")
# LoRA is a technique that freezes the pre-trained model weights and injects small, trainable
# "adapter" layers. This drastically reduces the number of parameters that need to be updated.
lora_config = LoraConfig(
    r=64,                           # The rank (or dimension) of the LoRA matrices. Higher can mean more expressive power but more parameters.
    lora_alpha=64,                  # A scaling factor for the LoRA updates.
    lora_dropout=0.05,              # Dropout probability for the LoRA layers to prevent overfitting.
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"], # Specific layers in the model to apply LoRA to.
    task_type="CAUSAL_LM"           # Specifies the task type for correct LoRA application.
)
# The `get_peft_model` function wraps the base model with the LoRA configuration.
model = get_peft_model(model, lora_config)

# 2.5: Print Trainable Parameters
logger.info("Trainable parameters after applying LoRA:")
# This function prints a summary of the model's parameters, highlighting the
# small percentage that are actually trainable, demonstrating the efficiency of LoRA.
model.print_trainable_parameters()

INFO:__main__:Loading processor from ./paligemma-3b-mix-224...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


INFO:__main__:Loading PaliGemma model in 8-bit precision...


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).



Loading checkpoint shards:   0%|                                                                                                           | 0/3 [00:00<?, ?it/s]

...


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:21<00:00,  7.06s/it]


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:21<00:00,  7.20s/it]


INFO:__main__:Configuring LoRA for efficient fine-tuning...


INFO:__main__:Trainable parameters after applying LoRA:


trainable params: 90,390,528 || all params: 3,013,857,008 || trainable%: 2.9992


In [3]:
# ==============================================================================
# Part 3: Data Preprocessing
# ==============================================================================

# 3.1: Define the Preprocessing Function
def preprocess_function(batch):
    questions = batch["problem"]
    images = batch["image"]
    answers = batch["solution"]

    processed_images = []
    texts_with_image = []

    for q, img in zip(questions, images):
        try:
            pil_img = img.convert("RGB").resize((224, 224))
            processed_images.append(pil_img)
            texts_with_image.append("<image> " + q)
        except Exception as e:
            logger.warning(f"Error processing an image, skipping it: {e}")
            processed_images.append(None)
            texts_with_image.append(None)

    valid_indices = [i for i, img in enumerate(processed_images) if img is not None]
    if not valid_indices:
        return {}

    processed_images = [processed_images[i] for i in valid_indices]
    texts_with_image = [texts_with_image[i] for i in valid_indices]
    valid_answers = [answers[i] for i in valid_indices]

    encoder_inputs = processor(
        images=processed_images,
        text=texts_with_image,
        padding="max_length",
        truncation=True,
        max_length=300,
        return_tensors="pt",
    )

    decoder_inputs = processor.tokenizer(
        text_target=valid_answers,
        padding="max_length",
        truncation=True,
        max_length=300,
        return_tensors="pt"
    )

    labels_ids = decoder_inputs["input_ids"].clone()
    padding_mask = decoder_inputs["attention_mask"] == 0
    labels_ids[padding_mask] = -100
    encoder_inputs["labels"] = labels_ids
    return encoder_inputs

# 3.2: Apply Preprocessing to the Datasets
logger.info("Applying preprocessing to the training dataset...")
processed_train_dataset = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)

logger.info("Applying preprocessing to the testing dataset...")
processed_test_dataset = test_dataset.map(
    preprocess_function, batched=True, remove_columns=test_dataset.column_names
)

INFO:__main__:Applying preprocessing to the training dataset...



Map:   0%|                                                                                                                      | 0/12600 [00:00<?, ? examples/s]


Map:   8%|████████▍                                                                                                  | 1000/12600 [00:34<06:42, 28.83 examples/s]


Map:   8%|████████▍                                                                                                  | 1000/12600 [00:46<06:42, 28.83 examples/s]


Map:  16%|████████████████▉                                                                                          | 2000/12600 [01:07<05:56, 29.76 examples/s]

...


INFO:__main__:Applying preprocessing to the testing dataset...



Map:   0%|                                                                                                                       | 0/1400 [00:00<?, ? examples/s]


Map:  71%|█████████████████████████████████████████████████████████████████████████████▏                              | 1000/1400 [00:22<00:09, 44.23 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:32<00:00, 42.46 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:36<00:00, 38.51 examples/s]

In [4]:
# ==============================================================================
# Part 4: Trainer Setup and Fine-Tuning
# ==============================================================================

# 4.1: Define Training Arguments
# This object holds all the settings and hyperparameters for the training run.
training_args = TrainingArguments(
    output_dir="./paligemma-clevr-finetuned", # Directory to save checkpoints and model outputs.
    num_train_epochs=1,                     # The total number of times to pass through the entire training dataset.
    per_device_train_batch_size=4,          # The number of training examples processed per GPU in one forward/backward pass.
    per_device_eval_batch_size=4,           # The batch size for evaluation.
    gradient_accumulation_steps=4,          # Accumulates gradients over multiple steps before performing an update. Effective batch size = 4 * 4 = 16. Helps with memory constraints.
    eval_strategy="steps",                  # Run evaluation at regular step intervals.
    eval_steps=100,                         # Evaluate the model every 100 training steps.
    save_strategy="steps",                  # Save a model checkpoint at regular step intervals.
    save_steps=100,                         # Save a checkpoint every 100 training steps.
    logging_steps=10,                       # Log training metrics (like loss) every 10 steps.
    learning_rate=1e-4,                     # The starting learning rate for the optimizer.
    load_best_model_at_end=True,            # After training, automatically load the checkpoint with the best evaluation score.
    report_to="none",                       # Disable reporting to external services like Weights & Biases.
    remove_unused_columns=False,            # Do not remove columns not used by the model (we've already done this).
    fp16=True,                              # Enable mixed-precision training to speed up training and save memory.
)

# 4.2: Initialize the Trainer
# The Trainer class abstracts away the complexity of the PyTorch training loop.
trainer = Trainer(
    model=model,                            # The PEFT-prepared model to be trained.
    args=training_args,                     # The training arguments we just defined.
    train_dataset=processed_train_dataset,  # The preprocessed training data.
    eval_dataset=processed_test_dataset,    # The preprocessed evaluation data.
    tokenizer=processor.tokenizer,          # The tokenizer for decoding predictions during evaluation.
)

# 4.3: Start the Fine-Tuning Process
logger.info("Starting the fine-tuning process...")
# This single command starts the entire training and evaluation loop.
trainer.train()
logger.info("Fine-tuning completed.")

/tmp/ipykernel_4082756/1548587602.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


INFO:__main__:Starting the fine-tuning process...


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.088500,0.087241
200,0.114900,0.077644
300,0.071800,0.068535
400,0.108600,0.048810
500,0.026900,0.027775
600,0.035200,0.042240
700,0.093900,0.023377


...

/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


INFO:__main__:Fine-tuning completed.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>